# 03 — Chat with your tiny SEC LLM

Run this **after `02_train_workshop.ipynb`** has finished training.

⚠️ **Important caveat:** the model you trained is a **base model** — pretrained on next-token prediction over SEC text, with no instruction tuning. It doesn't know how to "answer questions"; it knows how to *continue text*. So your prompts should look like the **start of an SEC passage**, not like a question.

| ❌ Bad prompt                    | ✅ Good prompt                                               |
|----------------------------------|--------------------------------------------------------------|
| `What are the risk factors?`     | `ITEM 1A. RISK FACTORS\n\nThe primary risks include`         |
| `Tell me about the business.`    | `Our business is`                                            |
| `Will the company succeed?`      | `We may be unable to`                                        |

## Cell 1 — Load your trained model

In [ ]:
import sys
sys.path.insert(0, '/workspace/zero-to-llm/pod')
from _inference import load_base_engine, complete

engine, tokenizer, meta, device = load_base_engine()
print(f'Model loaded on {device}.')
print(f'Vocab size: {tokenizer.get_vocab_size()}')
print(f'Layers    : {meta["model_config"]["n_layer"]}')
print(f'Embed dim : {meta["model_config"]["n_embd"]}')

## Cell 2 — Complete a single prompt

Edit `PROMPT` and run the cell. Edit again, run again.

In [ ]:
PROMPT       = 'ITEM 1A. RISK FACTORS\n\nThe following risks could materially affect our'
TEMPERATURE  = 0.8     # higher = more creative, lower = more conservative
TOP_K        = 50      # consider only the top-K most likely tokens at each step
MAX_TOKENS   = 200     # how long to generate

print(PROMPT, end='')
_ = complete(engine, tokenizer, PROMPT,
             max_tokens=MAX_TOKENS, temperature=TEMPERATURE, top_k=TOP_K)

## Cell 3 — Compare temperatures side by side

Run a single prompt with three different temperatures to feel how it changes output.

In [ ]:
PROMPT = 'Our principal sources of revenue are'

for t in [0.2, 0.7, 1.2]:
    print(f'\n=== temperature={t} ===')
    print(PROMPT, end='')
    complete(engine, tokenizer, PROMPT, max_tokens=80, temperature=t, top_k=50)

## Cell 4 — Multi-turn "conversation"

Since this is a base model, "multi-turn" means we keep appending the conversation as text and let the model continue. Run this cell, type a prompt, press Enter, type another, etc.

Type `:quit` (or interrupt the kernel) to stop.

In [ ]:
history = ''
while True:
    try:
        line = input('\n> ')
    except (EOFError, KeyboardInterrupt):
        print('\nbye.')
        break
    if line.strip() in (':quit', ':exit'):
        break
    history += line
    print(line, end='')
    out = complete(engine, tokenizer, history, max_tokens=120, temperature=0.7, top_k=50)
    history += out
    # Cap history so we don't blow context length
    history = history[-2000:]

## Cell 5 — Done

When you're done, **terminate the pod** to stop being billed:
[https://www.runpod.io/console/pods](https://www.runpod.io/console/pods)

If you saved the model in step 6 of the training notebook, you can download `/workspace/sec_llm.tar.gz` from the JupyterLab file browser before terminating.